In [5]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load the Binance dataset
df = pd.read_csv(r'/Users/utkugulbardak/Documents/training/Binance.csv')
df2 = pd.read_csv(r'/Users/utkugulbardak/Documents/training/BitMEX.csv')
df3 = pd.read_csv(r'/Users/utkugulbardak/Documents/training/Combined_Index.csv')
df4 = pd.read_csv(r'/Users/utkugulbardak/Documents/training/KuCoin.csv')
df5 = pd.read_csv(r'/Users/utkugulbardak/Documents/training/OKX.csv')

#print(df.head(10))

df.dtypes


ModuleNotFoundError: No module named 'torch'

In [2]:
# Convert 'Open time' and 'Close time' to datetime if they are in string format
df['Open time'] = pd.to_datetime(df['Open time'])
df['Close time'] = pd.to_datetime(df['Close time'])

# Display the first few rows to verify the changes
print(df.head())


            Open time  Open  High  Low  Close  Volume              Close time  \
0 2017-11-06 03:54:00   1.5  1.50  1.5   1.50   10.83 2017-11-06 03:54:59.999   
1 2017-11-06 03:55:00   1.3  1.30  1.3   1.30    1.00 2017-11-06 03:55:59.999   
2 2017-11-06 03:56:00   1.3  1.30  0.5   0.50   19.00 2017-11-06 03:56:59.999   
3 2017-11-06 03:57:00   0.5  0.61  0.5   0.61  253.00 2017-11-06 03:57:59.999   
4 2017-11-06 03:58:00   1.1  1.10  1.1   1.10   85.00 2017-11-06 03:58:59.999   

   Quote asset volume  Number of trades  Taker buy base asset volume  \
0              16.245               2.0                        10.83   
1               1.300               1.0                         0.00   
2              24.356               6.0                         0.33   
3             151.660              17.0                         0.00   
4              93.500               1.0                         0.00   

   Taker buy quote asset volume  Ignore  
0                        16.245     0.

In [3]:
# Convert 'Open time' to datetime (assuming 'Open time' is in seconds)
df['Open time'] = pd.to_datetime(df['Open time'], unit='s')

# Convert 'Close time' to datetime (assuming 'Close time' is in milliseconds)
df['Close time'] = pd.to_datetime(df['Close time'], unit='ms')

# Display the first few rows to verify the changes
print(df.head())


            Open time  Open  High  Low  Close  Volume              Close time  \
0 2017-11-06 03:54:00   1.5  1.50  1.5   1.50   10.83 2017-11-06 03:54:59.999   
1 2017-11-06 03:55:00   1.3  1.30  1.3   1.30    1.00 2017-11-06 03:55:59.999   
2 2017-11-06 03:56:00   1.3  1.30  0.5   0.50   19.00 2017-11-06 03:56:59.999   
3 2017-11-06 03:57:00   0.5  0.61  0.5   0.61  253.00 2017-11-06 03:57:59.999   
4 2017-11-06 03:58:00   1.1  1.10  1.1   1.10   85.00 2017-11-06 03:58:59.999   

   Quote asset volume  Number of trades  Taker buy base asset volume  \
0              16.245               2.0                        10.83   
1               1.300               1.0                         0.00   
2              24.356               6.0                         0.33   
3             151.660              17.0                         0.00   
4              93.500               1.0                         0.00   

   Taker buy quote asset volume  Ignore  
0                        16.245     0.

In [ ]:


# Create sequences and targets (for example, use the previous 10 rows to predict the next one)
sequence_length = 10
features = df[['open', 'high', 'low', 'close', 'volume']].values
targets = df['close'].shift(-1).dropna().values  # Predict the next day's 'close' price

# Prepare the input sequences and targets
sequences = []
for i in range(len(features) - sequence_length):
    sequences.append(features[i:i+sequence_length])
    
# Convert to tensors
sequences = torch.tensor(sequences, dtype=torch.float32)
targets = torch.tensor(targets[sequence_length:], dtype=torch.float32)

# Create a DataLoader
dataset = TensorDataset(sequences, targets)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Check one batch of data
for batch in dataloader:
    print(batch)
    break


ModuleNotFoundError: No module named 'torch'

In [38]:
import dash
from dash import html, dcc
import plotly.express as px
import pandas as pd

# Sample DataFrame (replace with your data)
df = pd.read_csv('data/populargames.csv')

# Ensure 'Release Date' is in the correct format for animation (numeric or date)
df['Release Date'] = pd.to_datetime(df['Release Date'], errors='coerce')

# Remove 'K' and other non-numeric characters, replace them with numeric values
df['Plays'] = df['Plays'].replace({r'[Kk]': 'e3'}, regex=True)  # Convert 'K' to 'e3' for thousands

# Convert 'Plays' to numeric, coercing any errors to NaN
df['Plays'] = pd.to_numeric(df['Plays'], errors='coerce')

# Drop rows with NaN in 'Plays'
df = df.dropna(subset=['Plays'])

# Create a unique ID for each game for better grouping
df['Game_ID'] = df['Title'] + "_" + df['Release Date'].dt.strftime('%Y-%m')

# Sort by 'Plays' to prepare for ranking
df = df.sort_values(by=['Release Date', 'Plays'], ascending=[True, False])

# Dynamically calculate the top 10 games over time
df['Cumulative Plays'] = df.groupby('Game_ID')['Plays'].cumsum()

# For each frame (month), select the top 10 games with the highest cumulative 'Plays'
top_10_over_time = df.groupby('Release Date', group_keys=False).apply(
    lambda x: x.nlargest(10, 'Cumulative Plays')
)

# Fill missing games to always have 10 rows
all_ranks = pd.DataFrame({
    "Rank": range(1, 11),
    "Game_ID": [""] * 10,
    "Cumulative Plays": [0] * 10,
    "Release Date": pd.Timestamp("1980-01-01")  # Placeholder for missing rows
})
frames = []
for date, group in top_10_over_time.groupby("Release Date"):
    group['Rank'] = group['Cumulative Plays'].rank(method="first", ascending=False)
    missing_ranks = 10 - group.shape[0]
    if missing_ranks > 0:
        fillers = all_ranks.copy()
        fillers["Release Date"] = date
        frames.append(pd.concat([group, fillers], ignore_index=True))

# Combine the frames into one consistent DataFrame
df_final = pd.concat(frames)

# Sort for consistent animation
df_final = df_final.sort_values(by=['Release Date', 'Rank'])

# Create a Plotly figure
fig = px.bar(df_final,
             x='Cumulative Plays',
             y='Rank',
             orientation='h',
             color='Game_ID',  # Assign a unique color per game
             animation_frame='Release Date',
             animation_group='Game_ID',
             range_x=[0, df_final['Cumulative Plays'].max() * 1.1],  # Extend x-axis for better visibility
             range_y=[0.5, 10.5],  # Fixed 10 bars
             title="Top 10 Games with the Most Plays Over Time",
             labels={"Rank": "Rank", "Cumulative Plays": "Plays", "Game_ID": "Game"},
             height=500
             )

# Update layout for consistent visuals
fig.update_layout(
    xaxis_title="Plays",
    yaxis_title=None,
    yaxis=dict(showticklabels=False),  # Remove rank labels
    legend_title="Game",  # Legend shows games
    margin=dict(l=50, r=50, t=50, b=50),
    transition_duration=2000  # Slow animation transitions
)

# Initialize Dash app
app = dash.Dash(__name__)

# Define layout
app.layout = html.Div([
    html.H1("Top 10 Games with the Most Plays Over Time"),
    dcc.Graph(figure=fig),
])

# Run the Dash app
if __name__ == '__main__':
    app.run_server(debug=True)


In [29]:
print(df.head())



      Unnamed: 0                Title Release Date  \
591          591              Pac-Man   1980-05-22   
1454        1454          Ms. Pac-Man   1982-02-03   
75            75    Super Mario Bros.   1985-09-13   
462          462  The Legend of Zelda   1986-02-21   
195          195  The Legend of Zelda   1986-02-21   

                                                   Team  Rating Times Listed  \
591             ['Wiz', 'Namco Networks America, Inc.']     3.4          427   
1454  ['Atari, Inc.', 'General Computer Corporation ...     3.6          198   
75                        ['Nintendo', 'Nintendo R&D4']     3.5         1.5K   
462                        ['Nintendo EAD', 'Nintendo']     3.2         1.2K   
195                        ['Nintendo EAD', 'Nintendo']     3.2         1.2K   

     Number of Reviews                     Genres  \
591                427                 ['Arcade']   
1454               198                 ['Arcade']   
75                1.5K  ['Adventure

In [19]:
print(df['Month'].unique())


['1980-05' '1982-02' '1985-09' '1986-02' '1986-06' '1986-09' '1987-01'
 '1987-07' '1987-12' '1988-08' '1988-10' '1989-05' '1989-06' '1989-11'
 '1990-07' '1990-09' '1990-10' '1990-11' '1991-02' '1991-03' '1991-06'
 '1991-07' '1991-08' '1991-10' '1991-11' '1991-12' '1992-04' '1992-10'
 '1992-11' '1992-12' '1993-03' '1993-06' '1993-07' '1993-08' '1993-09'
 '1993-11' '1993-12' '1994-03' '1994-04' '1994-08' '1994-09' '1994-10'
 '1994-11' '1995-02' '1995-03' '1995-04' '1995-08' '1995-11' '1995-12'
 '1996-01' '1996-02' '1996-03' '1996-05' '1996-06' '1996-09' '1996-10'
 '1996-11' '1996-12' '1997-01' '1997-03' '1997-06' '1997-07' '1997-08'
 '1997-10' '1997-11' '1997-12' '1998-01' '1998-06' '1998-07' '1998-09'
 '1998-10' '1998-11' '1998-12' '1999-01' '1999-02' '1999-04' '1999-05'
 '1999-08' '1999-09' '1999-10' '1999-11' '1999-12' '2000-02' '2000-03'
 '2000-04' '2000-06' '2000-07' '2000-10' '2000-11' '2000-12' '2001-02'
 '2001-03' '2001-05' '2001-06' '2001-07' '2001-08' '2001-09' '2001-10'
 '2001

In [21]:
print(df['Plays'].describe())


count     1509
unique     258
top        12K
freq        50
Name: Plays, dtype: object


In [31]:
df.head()

,Unnamed: 0,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,Month
591,591,Pac-Man,1980-05-22,"['Wiz', 'Namco Networks America, Inc.']",3.4,427,427,['Arcade'],Pac-Man is an arcade game developed by Namco a...,['Pacman is such a fucking idiot jesus christ....,5200.0,7,108,31,1980-05
1454,1454,Ms. Pac-Man,1982-02-03,"['Atari, Inc.', 'General Computer Corporation ...",3.6,198,198,['Arcade'],"In 1982, a sequel to the incredibly popular Pa...",['Do you really need a review of Ms. Pac- Man?...,1600.0,1,45,38,1982-02
75,75,Super Mario Bros.,1985-09-13,"['Nintendo', 'Nintendo R&D4']",3.5,1.5K,1.5K,"['Adventure', 'Platform']",A side scrolling 2D platformer and first entry...,"[""I actually had no idea this game was so long...",18000.0,59,733,237,1985-09
462,462,The Legend of Zelda,1986-02-21,"['Nintendo EAD', 'Nintendo']",3.2,1.2K,1.2K,['Adventure'],The Legend of Zelda is the first title in the ...,"['Managed to 100% this in less than two hours,...",9300.0,132,1.5K,463,1986-02
195,195,The Legend of Zelda,1986-02-21,"['Nintendo EAD', 'Nintendo']",3.2,1.2K,1.2K,['Adventure'],The Legend of Zelda is the first title in the ...,"['Managed to 100% this in less than two hours,...",9300.0,132,1.5K,463,1986-02
